### Day 1: Understand What Features You Need
  * Research what drives wildfire spread - wind, slope, vegetation, humidity, temperature
  * Identify which data sources provide each features
  * Plan your feature engineering approach

#### How wildfires start
  * Three elements are needed for a fire to start:
    * fuel (wood. brush, lichen)
    * oxygen (from the air)
    * ignition source (heat from lightning or human activities)
  
#### How wildfires spread?
  * The primary factors that influence the spread of wildfires are
    * Fuels
    * Weather
    * Topography (what the landscape in the area is like)

* Fuel - Vegetation type and density - what's available to burn - MODIS Land Cover or ESA WorldCover
* Weather - Wind speed, wind direction, temperature, humidity - Open Meteo API
* Topography - Elevation and slope - fire moves faster uphill - USGS or NASA SRTM elevation data

### Day 2: Get Weather Data
  * Sign up for the Open-Meto API - free, no API key required
  * Pull historical weather data for your Canadian wildfire case study location
  * Key variables: wind speed, wind direction, temperature, humidity

In [ ]:
import requests
import pandas as pd

In [ ]:
# temperature_2m - air temperature at 2 meters above the ground in degrees Celsius
# Higher temperatures dry out vegetation and accelerate spread.

# relative_humidity_2m - relative humidity at 2 meters above ground as a percentage
# Lower humidity means drier fuel which burns faster
latitude = 59.9
longitude = -119.8
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={latitude}&longitude={longitude}&start_date=2023-08-21&end_date=2023-09-30&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m"

In [ ]:
response = requests.get(url, timeout=60).json()

In [ ]:
hourly = response['hourly']

In [ ]:
weather_df = pd.DataFrame(hourly)

In [ ]:
weather_df['time'] = pd.to_datetime(weather_df['time'])

In [ ]:
weather_df.head()

In [ ]:
weather_df.to_csv("../data/weather_data.csv", index=False)

### Day 3: Get Terrain Data
  * Download elevation data for your case study region
  * Calculate slope from elevation - fire moves faster uphill
  * Source: USGS National Elevation Dataset or NASA SRTM

In [ ]:
# Step 1 - Define Your Bounding Box
  # You need to tell the elevation library which geographic area to download data for. Your bounding box is the rectangular region around your 
  # Canadian wildfire case study:
  # South boundary: latitude 58
  # North boundary: latitude 62
  # West boundary: longitude -122
  # East boundary: longitude -118

In [ ]:
south_boundary, north_boundary, west_boundary, east_boundary = 59, 61, -121, -119

In [ ]:
# Step 2 - Load your API key securely
  # using dotenv to read it from your .env file instead of hardcoding it

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENTOPOGRAPHY_API_KEY')

In [ ]:
# Step 3 - Call the OpenTopography API
  # pass your bounding box and API key as parameters using requests, same pattern as your Open-Meteo call

In [ ]:
import requests

In [ ]:
query_params = {
  "demtype": "SRTMGL1",
  "south": south_boundary,
  "north": north_boundary,
  "west": west_boundary,
  "east": east_boundary,
  "outputFormat": "GTiff",
  "API_Key": api_key
}

In [ ]:
response = requests.get(url="https://portal.opentopography.org/API/globaldem", params=query_params)

In [ ]:
# Step 4 - Save the response as a GeoTIFF file
  # the API returns raw binary data that you write directly to ../data/elevation.tif

In [ ]:
with open('../data/elevation.tif', "wb") as elevation_data_file:
    elevation_data_file.write(response.content)

In [ ]:
# Step 5 - Load the GeoTIFF using rasterio
  # read the elevation values into a NumPy array

In [ ]:
import rasterio # an open-source Python library used to read, write, and analyze geospatial raster data

In [ ]:
with rasterio.open('../data/elevation.tif') as elevation:
    print(elevation.nodata)
    
    # Read the elevation values from the first band into a NumPy array
    band1 = elevation.read(1)
    
    # Print the shape of the array to confirm it loaded correctly
    print(band1.shape)

In [ ]:
import numpy as np

In [ ]:
# Convert band1 to float64 first
band1_float = np.array(band1, dtype=float)

In [ ]:
(band1_float == elevation.nodata).sum()

In [ ]:
band1_float[band1_float == elevation.nodata] = np.nan

In [ ]:
np.isnan(band1_float).sum()

In [ ]:
band1_float

In [ ]:
gradient_north_south, gradient_east_west = np.gradient(band1_float) # computing the gradients of how the steep the terrain is across the north south and east west range respectively

In [ ]:
gradient_north_south_squared = gradient_north_south ** 2

In [ ]:
gradient_east_west_squared = gradient_east_west ** 2

In [ ]:
slope = np.sqrt(gradient_north_south_squared + gradient_east_west_squared)

In [ ]:
np.nanmin(slope), np.nanmax(slope)

In [ ]:
# Step 7 - Visualize
  # Plot both the elevation and slope as heatmaps uisng Matplotlib so you can see the terrain of your case study region visually.
  # Mountainous areas will show up clearly

In [ ]:
# Slope - using slope with a YlOrRd colormap
# Plot 2 - Slope
  # A 2D grid where each pixel represents how steep the terrain is at that location. Flat areas will
  # be close to zero. Steep mountain faces will have high values. This is the feature that directly
  # influences how fast fire spreads - steep terrain means faster spread

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2)

# Plotting the elevations of the specified boundary above on a terrain colormap
terrain_color_map = ax[0].imshow(band1_float, cmap='terrain')
plt.colorbar(terrain_color_map) # plotting the range of elevations throughout the specified boundary
ax[0].set_title("Elevation (meters)")

# Plotting the 3D slopes of the terrain across all directions 
slope_color_map = ax[1].imshow(slope, cmap='YlOrRd')
plt.colorbar(slope_color_map) # mostly flat land with very little changes in elevation
ax[1].set_title("Slope")

fig.suptitle("Terrain Analysis")

plt.tight_layout()
plt.show()

In [ ]:
# Step 8 - Save both the elevation and slope arrays so you can merge them with your fire detection
np.save('../data/elevation.npy', band1_float)
np.save('../data/slope.npy', slope)

### Day 4: Get Vegetation Data
  * Download land cover / vegetation type data
  * Source: ESA WorldCover
  * Different vegetation types burn at different rates and intensities

In [ ]:
# 1, Sign up for the Copernicus Data Space at dataspace.copernicus.eu - free acount, gives you access to ESA datasets
# 2. Find the WorldCover tile covering your bounding box
# 3. Download the GeoTIFF for that tile
# 4. Load it with rasterio - same approach as your elevation data
# 5. Visualize it - each pixel value represents a land cover class
# 6. Save it as a .npy file for merging later

In [ ]:
# The vegetation.tif contains a 2D grid where every pizel represents a land cover classification for a 10x10 meter area on the ground.
# Each pixel has a numeric value that corresponds to one of 11 land cover types:
  # 10 - Trees
  # 20 - Shrubland
  # 30 - Grassland
  # 40 - Cropland
  # 50 - Built-up areas
  # 60 - Bare/sparse vegetation
  # 70 - Snow and ice
  # 80 - Permanent water bodies
  # 90 - Herbaceous wetland
  # 100 - Moss and lichen

In [ ]:
from rasterio.merge import merge

In [ ]:
with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N57W123_Map.tif') as vegetation_south:
  with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W120_Map.tif') as vegetation_east:
    with rasterio.open('../data/ESA_WorldCover_10m_2021_v200_N60W123_Map.tif') as vegetation:
      merged_array, merged_transform = merge([vegetation, vegetation_east, vegetation_south])

In [ ]:
merged = merged_array[0]

In [ ]:
merged.shape

In [ ]:
vegetation_row_start, vegetation_column_start = rasterio.transform.rowcol(merged_transform, west_boundary, north_boundary)

In [ ]:
vegetation_row_start, vegetation_column_start

In [ ]:
vegetation_row_end, vegetation_column_end = rasterio.transform.rowcol(merged_transform, east_boundary, south_boundary)

In [ ]:
vegetation_row_end, vegetation_column_end

In [ ]:
vegetation_clipped = merged[vegetation_row_start:vegetation_row_end, vegetation_column_start:vegetation_column_end]

In [ ]:
vegetation_clipped.shape

In [ ]:
np.save('../data/vegetation.npy', vegetation_clipped)

### Day 5: Join All Data Sources
  * Merge weather, terrain, and vegetation data onto your fire detection dataframe
  * Each fire detection row should now have wind, slope, and vegetation attributes

In [ ]:
# 1. Filtered your cleaned fire dataframe to your case study region - latitude 59-61, longitude -121 to -119, weeks 34 to 30

In [ ]:
cleaned_fire_df = pd.read_csv('../data/cleaned_fire_archive.csv')

In [ ]:
cleaned_fire_df['acq_date'] = pd.to_datetime(cleaned_fire_df['acq_date'], format="%Y-%m-%d")

In [ ]:
cleaned_fire_df['week'] = cleaned_fire_df['acq_date'].dt.isocalendar().week

In [ ]:
filtered_clean_fire_df = cleaned_fire_df[(cleaned_fire_df['latitude'].between(59, 61)) & (cleaned_fire_df['longitude'].between(-121, -119)) & (cleaned_fire_df['week'].between(34, 39))]

In [ ]:
# 2. Join weather data - match each fire detection to the closest hourly weather reading by timestamp

In [ ]:
weather_df = pd.read_csv('../data/weather_data.csv')
weather_df

In [ ]:
weather_df['time'] = pd.to_datetime(weather_df['time'])

In [ ]:
filtered_clean_fire_df['acq_time'] = filtered_clean_fire_df['acq_time'].astype(str)

In [ ]:
filtered_clean_fire_df['acq_time'] = filtered_clean_fire_df['acq_time'].str.zfill(4)

In [ ]:
filtered_clean_fire_df['acq_time']

In [ ]:
filtered_clean_fire_df['acq_date'] = filtered_clean_fire_df['acq_date'].dt.strftime('%Y-%m-%d')

In [ ]:
filtered_clean_fire_df['datetime'] = filtered_clean_fire_df['acq_date'] + " " + filtered_clean_fire_df['acq_time']

In [ ]:
filtered_clean_fire_df['datetime'] = pd.to_datetime(filtered_clean_fire_df['datetime'])

In [ ]:
filtered_clean_fire_df['datetime'] = filtered_clean_fire_df['datetime'].dt.round('h')

In [ ]:
filtered_clean_fire_df['datetime']

In [ ]:
filtered_clean_fire_df.head()

In [ ]:
weather_df = weather_df.rename(columns={'time':'datetime'})

In [ ]:
weather_df

In [ ]:
merged_fire_df = pd.merge(filtered_clean_fire_df, weather_df, on='datetime', how='left')

In [ ]:
merged_fire_df.shape

In [ ]:
merged_fire_df.columns.tolist()

In [ ]:
# 3. Join elevation and slope - for each fire detection latitude/longitude, look up the 
# corresponding pixel value in your elevation and slope arrays

In [ ]:
# Open your elevation.tif with rasterio and get its transform object
# For each row in merged_fire_df, use rasterio.transform.rowcol() to convert the latitude and longitude to row and column indicies in the elevation array
# Use those indices to look up the elevation value from your band1_float array
# Do the same for your slope array -- same indices, different array
# Add both as new columns to merged_fire_df - call them elevation and slope

In [ ]:
import rasterio

In [ ]:
with rasterio.open('../data/elevation.tif') as elevation:
    rows, cols = rasterio.transform.rowcol(elevation.transform, merged_fire_df['longitude'], merged_fire_df['latitude'])

In [ ]:
np.clip(rows, 0, 7199, out=rows)
np.clip(cols, 0, 7199, out=cols)

In [ ]:
band1_float[rows, cols]

In [ ]:
slope[rows, cols]

In [ ]:
merged_fire_df['elevation'] = band1_float[rows, cols] # add elevation values to the dataframe

In [ ]:
merged_fire_df['slope'] = slope[rows, cols] # add slope values to the dataframe

In [ ]:
merged_fire_df.head()

In [ ]:
# 4. Join vegetation -- same approach, look up the pixel value for each fire detection location
vegetation_rows, vegetation_cols = rasterio.transform.rowcol(merged_transform, merged_fire_df['longitude'], merged_fire_df['latitude'])  
vegetation_rows.min(), vegetation_rows.max(), vegetation_cols.min(), vegetation_cols.max()

In [ ]:
np.clip(vegetation_rows, 0, 23999, out=vegetation_rows)
np.clip(vegetation_cols, 0, 23999, out=vegetation_cols)

In [ ]:
merged_fire_df["vegetation"] = vegetation_clipped[vegetation_rows, vegetation_cols]

In [ ]:
# 5. Check for missing values in the merged dataframe

In [ ]:
merged_fire_df.isnull().sum()

In [ ]:
merged_fire_df['elevation'] = merged_fire_df['elevation'].fillna(merged_fire_df['elevation'].mean())

In [ ]:
merged_fire_df['slope'] = merged_fire_df['slope'].fillna(merged_fire_df['slope'].mean())

In [ ]:
merged_fire_df.isnull().sum()

In [ ]:
# 6. Save the merged dataframe as merged_fire_data.csv in your data/ folder
merged_fire_df.to_csv('../data/merged_fire_data.csv', index=False)

### Day 6: Feature Engineering
  * Create derived features - wind alignment with slope, days since last rain, vegetation dryness index
  * These compound features often matter more than raw variables

#### 1. Wind alignment with slope
  * fire spreads fastest when wind blows uphill. Create a feature that combines wind direction and slope to capture this interaction. 

In [ ]:
# 1. Convert wind_direction_10m from degrees to radians using np.deg2rad()
wind_direction_rad = np.deg2rad(merged_fire_df['wind_direction_10m'])

In [ ]:
# 2. Calculate the east-west wind component using np.cos() on the radians
east_west_wind = np.cos(wind_direction_rad)

In [ ]:
# 3. Calculate the north-south wind component using np.sin() on the radians
north_south_wind = np.sin(wind_direction_rad)

In [ ]:
# 4. Multiply the wind speed by each component to get directional wind vectors

In [ ]:
east_west_wind_speed = east_west_wind * merged_fire_df['wind_speed_10m']

In [ ]:
north_south_wind_speed = north_south_wind * merged_fire_df['wind_speed_10m']

In [ ]:
# 5. Multiply those by slope to get a wind-slope alignment score

In [ ]:
east_west_wind = east_west_wind_speed * merged_fire_df['slope']
north_south_wind = north_south_wind_speed * merged_fire_df['slope']

In [ ]:
wind_slope_alignment = east_west_wind + north_south_wind

In [ ]:
# 6. Save the result as a new column called wind_slope_alignment

In [ ]:
merged_fire_df['wind_slope_alignment'] = wind_slope_alignment

#### 2. Vapor Pressure Deficit (VPD)
  * a measure of how much moisture the air can still absorb. High VPD means dry conditions that accelerate fire spread. 
  * Calculate it from temperature and relative humidity

In [ ]:
# 1. Saturation vapor pressure - the maximum amount of moisture air can hold at a given temperature. 
# Formula: 0.6108 * np.exp(17.27 * T / (T + 273.3)) where T is temperature in Celsius

In [ ]:
merged_fire_df.columns.tolist()

In [ ]:
temperatures = merged_fire_df['temperature_2m']

In [ ]:
saturation_vapor_pressure = 0.6108 * np.exp(17.27 * temperatures / (temperatures + 237.3))

In [ ]:
# 2. Actual vapor pressure - how much moisture the air actually contains. 
# Formula: saturation vapor pressure multiplied by relative humidity divided by 100

In [ ]:
humidity = merged_fire_df['relative_humidity_2m']

In [ ]:
actual_vapor_pressure = (saturation_vapor_pressure * humidity) / 100

In [ ]:
# 3. VPD - saturation vapor pressure minus actual vapor pressure

In [ ]:
merged_fire_df['vpd'] = saturation_vapor_pressure - actual_vapor_pressure

#### 3. Fuel dryness proxy
  * combine temperature and humidity into a single dryness score.
  * Higher temperature and lower humidity means drier fuel.

In [ ]:
merged_fire_df['fuel_dryness'] = temperatures / humidity

In [ ]:
merged_fire_df['fuel_dryness']

#### 4. Fire spread risk score
  * combine FRP, slope, wind speed, and fuel dryness into a single risk score

In [ ]:
merged_fire_df.columns.tolist()

In [ ]:
fire_radiative_power = merged_fire_df['frp'] # Fire Radiative Power - Energy Output of each Fire Detection

In [ ]:
wind_speed_10m = merged_fire_df['wind_speed_10m'] # standard wind speed measured at 10 meters above the ground

In [ ]:
slope = merged_fire_df['slope'] # what the 3D slope at every point in the scanned boundary looks like

In [ ]:
fuel_dryness = merged_fire_df['fuel_dryness'] 

In [ ]:
fire_radiative_power

In [ ]:
wind_speed_10m

In [ ]:
slope

In [ ]:
fuel_dryness

In [ ]:
merged_fire_df['fire_spread_risk'] = fire_radiative_power * wind_speed_10m * slope * fuel_dryness

#### 5. Verify your new features
  * print summary statistics on all new columns to confirm they look reasonable

In [ ]:
merged_fire_df[['wind_slope_alignment', 'vpd', 'fuel_dryness', 'fire_spread_risk']].describe()

### Day 7 - Wrap Up Week 2
  * Commit everything to GitHub
  * Document your feature set clearly in a notebook markdown cell

In [ ]:
merged_fire_df.to_csv('../data/merged_fire_data.csv', index=False)

### Summary
* In Week 2, I feature engineered my dataset as follows:
    * I research and identified how fires start and spread by figuring out which factors are the root cause
    * I fetched the weather data for the location where the fire I had selected from the end of Part 1 from Open-Meteo Weather API during late August - September time frame
    * I obtained elevation data from Open Topography API to be able to create new columns for elevation and slope for every point in the specified fire boundary
    * I downloaded vegetation data for the specified geographical location
    * I merged the weather, terrain, and vegetation data onto the fire detection dataframe from Part 1
    * I feature engineered 4 new columns - wind slope adjustment, vapor pressure deficit, fuel dryness proxy, and fire spread risk score